# 02 – Transaction Monitoring Rules
Rules are written in `/sql` and executed here. Labels (`Is_laundering`, `Laundering_type`) are **never** used inside rules, only for evaluation in notebook 03.

In [4]:
import duckdb
import pandas as pd

con = duckdb.connect("../aml.duckdb")
con.sql("SET enable_progress_bar = false")

def run_sql(path):
    """Read a .sql file and run it against the database."""
    with open(path) as f:
        con.sql(f.read())
    print(f"Ran {path}")

In [5]:
con.sql("""
SELECT COUNT(*) AS n,
       SUM(CASE WHEN Sender_account = Receiver_account THEN 1 ELSE 0 END) AS same_account,
       COUNT(DISTINCT Sender_account)   AS n_senders,
       COUNT(DISTINCT Receiver_account) AS n_receivers
FROM tx
WHERE Payment_type = 'Cash Deposit'
""").df()

,n,same_account,n_senders,n_receivers
0,225206,0.0,24328,15179


In [6]:
con.sql("""
WITH weekly AS (
    SELECT Receiver_account          AS account,
           date_trunc('week', ts)    AS wk,
           COUNT(*)                  AS n_deposits,
           SUM(Amount)               AS total_deposited
    FROM tx
    WHERE Payment_type = 'Cash Deposit'
    GROUP BY account, wk
)
SELECT quantile_cont(n_deposits, 0.50)      AS p50_count,
       quantile_cont(n_deposits, 0.99)      AS p99_count,
       MAX(n_deposits)                      AS max_count,
       quantile_cont(total_deposited, 0.99) AS p99_weekly_total
FROM weekly
""").df()

,p50_count,p99_count,max_count,p99_weekly_total
0,1.0,3.0,9,9282.1196


## R1 – Structuring / smurfing

In [7]:
run_sql("../sql/r1_structuring.sql")

con.sql("""
SELECT COUNT(*)                         AS alert_weeks,
       COUNT(DISTINCT account)          AS accounts_alerted,
       ROUND(AVG(n_deposits), 1)        AS avg_deposits,
       ROUND(AVG(n_depositors), 1)      AS avg_depositors,
       ROUND(AVG(total_deposited), 0)   AS avg_weekly_total
FROM r1_alerts
""").df()

Ran ../sql/r1_structuring.sql


CatalogException: Catalog Error: Table with name r1_alerts does not exist!
Did you mean "pg_tables"?